# Gaussian processes: Bayesian uncertainty over functions

**Companion to:** `bayes_29_07_2026.tex`, §*Random functions, Gaussian processes and
kernels* (`sec:bayes-modelling`).

## Historical context

Every earlier notebook places a prior over a fixed, finite set of numbers (a
probability $\theta$, a state vector). The chapter's "modelling revolution" section
describes a further generalisation: placing a prior over an entire **function**. George
Kimeldorf and Grace Wahba established the mathematical correspondence between Bayesian
estimation of random functions and spline smoothing:

> Kimeldorf, G., & Wahba, G. (1970). *A Correspondence Between Bayesian Estimation on
> Stochastic Processes and Smoothing by Splines*. The Annals of Mathematical
> Statistics, 41(2), 495–502.

Anthony O'Hagan then developed an explicitly Bayesian curve-fitting framework not
restricted to a fixed finite-dimensional form:

> O'Hagan, A. (1978). *Curve Fitting and Optimal Design for Prediction*. Journal of the
> Royal Statistical Society, Series B, 40(1), 1–42.

A **Gaussian process (GP)** places a joint Gaussian distribution over the function's
values at any finite set of input points, with a **kernel** encoding how correlated
those values are. This notebook fits a GP to a genuinely famous real dataset — the
Mauna Loa atmospheric CO2 record — the same dataset used as the canonical GP worked
example in Rasmussen & Williams' textbook.

Further reading: Rasmussen, C. E., & Williams, C. K. I. (2006), *Gaussian Processes for
Machine Learning*, MIT Press (freely available at gaussianprocess.org/gpml), Ch. 5,
§5.4.3.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, WhiteKernel, ConstantKernel, ExpSineSquared, RationalQuadratic,
)

from utils.plotting import set_style, save_fig, PALETTE

set_style()
NB_DIR = pathlib.Path.cwd()
rng_seed = 4

## Problem & data (real)

The Mauna Loa Observatory has recorded atmospheric CO2 concentration continuously since
1958 (the "Keeling Curve"). It's bundled with `statsmodels` so it loads offline, with no
network fetch needed.

In [ ]:
from statsmodels.datasets import co2

raw = co2.load_pandas().data  # weekly CO2 ppm, DatetimeIndex, some missing weeks
monthly = raw["co2"].resample("MS").mean().interpolate()  # monthly, gaps filled

years = monthly.index.year + (monthly.index.dayofyear - 1) / 365.25
X = years.values.reshape(-1, 1)
y = monthly.values

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(years, y, color=PALETTE["true"], lw=1)
ax.set_title("Mauna Loa atmospheric CO2 (real data)")
ax.set_xlabel("year")
ax.set_ylabel("CO2 (ppm)")
fig.tight_layout()
save_fig(fig, "co2_raw_data", NB_DIR)
plt.show()

print(f"{len(y)} monthly observations, {years.min():.1f} to {years.max():.1f}")

## Model: a composite kernel

CO2 has a smooth long-term rising trend, a strong yearly seasonal cycle, and irregular
medium-term wiggles — a good showcase for **kernel composition**, since a GP kernel
built by adding/multiplying simpler kernels is still a valid kernel:

$$
k = \underbrace{k_{\text{trend}}}_{\text{long-term rise, RBF}}
  + \underbrace{k_{\text{seasonal}}}_{\text{RBF} \times \text{periodic, period=1yr}}
  + \underbrace{k_{\text{irregular}}}_{\text{rational quadratic}}
  + \underbrace{k_{\text{noise}}}_{\text{white noise}}.
$$

This is essentially the kernel used in the standard scikit-learn/GPML Mauna Loa
worked example. We first look at what the *prior* (before seeing data) believes
functions look like, then condition on the real observations to get the posterior.

In [ ]:
k_trend = ConstantKernel(60.0) * RBF(length_scale=50.0)
k_seasonal = (
    ConstantKernel(2.0)
    * RBF(length_scale=100.0)
    * ExpSineSquared(length_scale=1.0, periodicity=1.0, periodicity_bounds="fixed")
)
k_irregular = ConstantKernel(0.5) * RationalQuadratic(length_scale=1.0, alpha=1.0)
k_noise = WhiteKernel(noise_level=0.1)

kernel = k_trend + k_seasonal + k_irregular + k_noise

In [ ]:
# --- Prior draws, before seeing any data ---
X_grid_prior = np.linspace(years.min(), years.min() + 8, 200).reshape(-1, 1)
gp_prior = GaussianProcessRegressor(kernel=kernel, random_state=rng_seed)
prior_mean, prior_std = gp_prior.predict(X_grid_prior, return_std=True)
prior_samples = gp_prior.sample_y(X_grid_prior, n_samples=5, random_state=rng_seed)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(X_grid_prior, prior_samples, color=PALETTE["muted"], lw=1, alpha=0.8)
ax.fill_between(X_grid_prior.ravel(), prior_mean - 2 * prior_std, prior_mean + 2 * prior_std,
                 color=PALETTE["muted"], alpha=0.15)
ax.set_title("GP prior: function draws before seeing any CO2 data")
ax.set_xlabel("year")
ax.set_ylabel("CO2 (ppm), centred")
fig.tight_layout()
save_fig(fig, "gp_prior_draws", NB_DIR)
plt.show()

## Fit: condition the GP on the real data, and extrapolate forward

`GaussianProcessRegressor.fit` optimises the kernel's hyperparameters (length scales,
noise level) by maximising the marginal likelihood — the same "evidence" quantity that
appears as the normalising constant in Bayes' theorem throughout this chapter.

In [ ]:
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=3,
                               random_state=rng_seed)
gp.fit(X, y)
print("Fitted kernel:", gp.kernel_)

# Extrapolate 20 years beyond the observed record
X_future = np.linspace(years.min(), years.max() + 20, 800).reshape(-1, 1)
mean_pred, std_pred = gp.predict(X_future, return_std=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years, y, color=PALETTE["true"], lw=1.2, label="observed CO2 (real data)")
ax.plot(X_future, mean_pred, color=PALETTE["approx"], lw=1.5, label="GP posterior mean")
ax.fill_between(X_future.ravel(), mean_pred - 2 * std_pred, mean_pred + 2 * std_pred,
                 color=PALETTE["approx"], alpha=0.2, label="95% credible band")
ax.axvline(years.max(), color=PALETTE["muted"], ls="--", lw=1)
ax.set_title("GP fit to Mauna Loa CO2, extrapolated 20 years beyond the data")
ax.set_xlabel("year")
ax.set_ylabel("CO2 (ppm)")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
save_fig(fig, "gp_posterior_extrapolation", NB_DIR)
plt.show()

## Takeaways

- The GP prior alone already encodes real, useful structure — smoothness, and (through
  the periodic kernel factor) a built-in yearly rhythm — before it has seen a single
  data point.
- After conditioning on the real Mauna Loa record, the posterior mean tracks both the
  long-term rise and the seasonal oscillation, and its uncertainty band widens exactly
  where it should: beyond the last observed year, where the model has no data to
  constrain it. This is the concrete meaning of "Bayesian uncertainty over functions" —
  the whole curve, not just a handful of parameters, carries a probability distribution.
- This is the same conceptual object the chapter connects to modern kernel-based
  machine learning; the next notebook uses exactly this fitted-GP machinery as the
  **surrogate model** inside Bayesian optimisation.